# ²⁹SiO⁺ X²Σ⁺ Zeeman Crossing Map

**Task 3 exploration — N=0(+)/N=1(−) level crossings near 1.5 T**

## Model summary

Molecule: ²⁹SiO⁺, electronic ground state X²Σ⁺ (v=0).  
Nuclear spins: ²⁹Si I=½, ¹⁶O I=0 → `I_nuclei=[0, 1/2]`, `P_values=[1/2]`.  
Included rotational levels: N=0, 1, 2 → **36 Zeeman sublevels** total.

### Constants provenance

| Constant | Value (MHz) | Source |
|----------|------------|--------|
| B_rot | ~21 245 | Zhu 2022 (derived from B_e/r_e; see `molecule_parameters.py`) |
| γ (spin-rotation) | 12 MHz | Zhu 2022 Tab. 1 |
| b_F (Fermi contact) | — | Knight 1985 (²⁹Si hyperfine) |
| c (dipolar hf) | — | Knight 1985 |
| g_S | 2.002319 | standard (free electron) |
| μ_E | — | Zhu 2022 |

**Conventions:** Brown & Carrington (B&C) audited; see `docs/sio-conventions.md`. Parity convention +(-1)^N verified against the code's own `Parity_mat` in `test_sio_crossing.py` gate 1.  
The N=0→N=1 interval (~42.49 GHz) divided by the differential Zeeman slope (~2.80 MHz/G) predicts the crossing near **~15 170 G = 1.517 T** — consistent with Karthein's 1.505–1.530 T window.

### Physics mechanism (Karthein)

At high field, N=0 sublevels with m_S=+½ rise at +g_S μ_B/2 ≈ +1.40 MHz/G while N=1 sublevels with m_S=−½ descend. The differential slope ~2.80 MHz/G tunes the two manifolds into degeneracy. A static axial E-field (ΔM_F=0) couples the crossing pair through the m_S admixture in each high-field eigenstate — the coupling is suppressed relative to a bare-dipole drive by a factor ξ ~ 2.7×10⁻⁴.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1 — Setup: sys.path, imports, model construction
# ---------------------------------------------------------------------------
import os, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['font.size'] = 11

# --- sys.path: walk up to repo root, then add 'Source Code' ---
_nb_dir = os.path.abspath('')
_root = _nb_dir
while _root and 'Source Code' not in os.listdir(_root):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        raise RuntimeError('Could not find Source Code directory')
    _root = _parent
_src = os.path.join(_root, 'Source Code')
if _src not in sys.path:
    sys.path.insert(0, _src)
print(f'Source Code path: {_src}')

from Energy_Levels import MoleculeLevels
print('MoleculeLevels imported OK')

# --- figures directory (notebook-local) ---
FIG_DIR = os.path.join(_nb_dir, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Figures directory: {FIG_DIR}')

# --- Instantiate model ---
M = MoleculeLevels.initialize_state(
    'SiO+', 'X', 0,
    N_list=[0, 1, 2],
    fermion_or_boson='boson',
    I_nuclei=[0, 1/2],
    P_values=[1/2]
)
print(f'Model: SiO+ X2Sigma+ v=0, N=[0,1,2], size={M.size} states (expect 36)')
assert M.size == 36, f'Expected 36 states, got {M.size}'

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 — Shared analysis (computed once, reused by all figures)
#
# Coarse map (500 pts) for Fig 1.
# Crossing identification on a medium map (1200 pts, full scan) for labeling.
# Refined map (600 pts, zoom) for Fig 2.
# ---------------------------------------------------------------------------

# --- Helper functions (same logic as gate script) ---
def zeeman_map(m, B):
    evB, vecB = m.ZeemanMap(B, output=True, write_attribute=False, order=True)
    return np.array(evB), np.array(vecB)

def dom_N(m, vec):
    return m.q_numbers['N'][int(np.argmax(vec**2))]

def dom_M(m, vec):
    return m.q_numbers['M'][int(np.argmax(vec**2))]

def parity_of(m, vec):
    return int(np.round(vec @ m.Parity_mat @ vec))

def find_crossings(m, B, evB, vecB, n0_idx, n1_idx):
    out = []
    for a in n0_idx:
        for b in n1_idx:
            gap = evB[:, b] - evB[:, a]
            sc = np.where(np.diff(np.sign(gap)) != 0)[0]
            for i in sc:
                f = gap[i] / (gap[i] - gap[i+1])
                Bc = B[i] + f * (B[i+1] - B[i])
                slope = abs((gap[i+1] - gap[i]) / (B[i+1] - B[i]))
                gmin = min(abs(gap[i]), abs(gap[i+1]))
                pa = parity_of(m, vecB[i, a])
                pb = parity_of(m, vecB[i, b])
                Ma = dom_M(m, vecB[i, a])
                Mb = dom_M(m, vecB[i, b])
                out.append(dict(Bc=Bc, a=a, b=b, idx=i, slope=slope, gmin=gmin,
                                pa=pa, pb=pb, Ma=Ma, Mb=Mb))
    out.sort(key=lambda r: r['Bc'])
    return out

def refine_crossing(m, c):
    B = np.linspace(c['Bc'] - 30, c['Bc'] + 30, 601)
    ev, vec = zeeman_map(m, B)
    gap = ev[:, c['b']] - ev[:, c['a']]
    sc = np.where(np.diff(np.sign(gap)) != 0)[0]
    if len(sc) == 0:
        return c
    i = sc[len(sc) // 2]
    f = gap[i] / (gap[i] - gap[i+1])
    c = dict(c)
    c['Bc'] = B[i] + f * (B[i+1] - B[i])
    c['slope'] = abs((gap[i+1] - gap[i]) / (B[i+1] - B[i]))
    c['gmin'] = min(abs(gap[i]), abs(gap[i+1]))
    return c

def evec_offcrossing(m, Bc, ref_vecs, a, b, offset=15.0):
    _, vec_off = m.eigensystem(0.0, Bc - offset)
    ov = np.abs(vec_off @ ref_vecs.T)
    ka = int(np.argmax(ov[:, a]))
    kb = int(np.argmax(ov[:, b]))
    return vec_off[ka], vec_off[kb]

def zero_field_label(m, vecB, k):
    v = vecB[0, k]
    idx = int(np.argmax(v**2))
    return {q: m.q_numbers[q][idx] for q in ['N', 'J', 'F', 'M']}, v[idx]**2

# --- Coarse full scan (500 pts) for Fig 1 ---
SCAN_LO, SCAN_HI = 14000.0, 16000.0
WINDOW_LO, WINDOW_HI = 15050.0, 15300.0
B_COARSE = np.linspace(1e-6, SCAN_HI, 500)
EV_C, VEC_C = zeeman_map(M, B_COARSE)
print(f'Coarse map: {len(B_COARSE)} points, 0 to {SCAN_HI:.0f} G')

# Track zero-field N character
SIZE = M.size
DOMN0 = np.array([dom_N(M, VEC_C[0, k]) for k in range(SIZE)])
N0_IDX = [k for k in range(SIZE) if DOMN0[k] == 0]
N1_IDX = [k for k in range(SIZE) if DOMN0[k] == 1]
N2_IDX = [k for k in range(SIZE) if DOMN0[k] == 2]
print(f'N=0: {len(N0_IDX)} states, N=1: {len(N1_IDX)} states, N=2: {len(N2_IDX)} states')

# --- Medium scan (1200 pts) for crossing identification only ---
B_MED = np.linspace(1e-6, SCAN_HI, 1200)
EV_M, VEC_M = zeeman_map(M, B_MED)
print(f'Medium map: {len(B_MED)} points (for crossing search)')

DOMN0_M = np.array([dom_N(M, VEC_M[0, k]) for k in range(SIZE)])
N0_IDX_M = [k for k in range(SIZE) if DOMN0_M[k] == 0]
N1_IDX_M = [k for k in range(SIZE) if DOMN0_M[k] == 1]

ALL_CROSS = find_crossings(M, B_MED, EV_M, VEC_M, N0_IDX_M, N1_IDX_M)
OPP = [c for c in ALL_CROSS if c['pa'] != c['pb']]
# Refine each opposite-parity crossing
OPP = [refine_crossing(M, c) for c in OPP]
IN_WINDOW = [c for c in OPP if WINDOW_LO <= c['Bc'] <= WINDOW_HI]
DM0 = [c for c in IN_WINDOW if c['Ma'] == c['Mb']]
CANON = min(DM0, key=lambda c: abs(c['Bc'] - 15170.0)) if DM0 else None

print(f'\nOpposite-parity N0xN1 crossings found: {len(OPP)}')
print(f'  In Karthein window [{WINDOW_LO:.0f}, {WINDOW_HI:.0f}] G: {len(IN_WINDOW)}')
print(f'  DeltaM_F=0 in window: {len(DM0)}')
if CANON:
    print(f'  Canonical crossing: Bc = {CANON["Bc"]:.1f} G')

# --- Refined zoom scan (600 pts) for Fig 2 ---
ZOOM_LO, ZOOM_HI = 15050.0, 15300.0
B_ZOOM = np.linspace(ZOOM_LO, ZOOM_HI, 600)
EV_Z, VEC_Z = zeeman_map(M, B_ZOOM)
print(f'\nZoom map: {len(B_ZOOM)} points, {ZOOM_LO:.0f}–{ZOOM_HI:.0f} G')

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3 — Identify canonical pair and compute crossing energy for Fig 2 window
# ---------------------------------------------------------------------------
assert CANON is not None, 'No canonical Delta M_F=0 crossing found in window'

la, wa = zero_field_label(M, VEC_M, CANON['a'])
lb, wb = zero_field_label(M, VEC_M, CANON['b'])

print('=== Canonical crossing pair ===')
print(f'  Bc = {CANON["Bc"]:.1f} G = {CANON["Bc"]/1e4:.4f} T')
print(f'  Partner A (parity {CANON["pa"]:+d}): '
      f'|N={la["N"]:.0f}, J={la["J"]}, F={la["F"]:.0f}, M_F={la["M"]:+.0f}> (w={wa:.3f})')
print(f'  Partner B (parity {CANON["pb"]:+d}): '
      f'|N={lb["N"]:.0f}, J={lb["J"]}, F={lb["F"]:.0f}, M_F={lb["M"]:+.0f}> (w={wb:.3f})')

# Crossing energy (average of two partners at canonical Bc on medium map)
idx_bc = CANON['idx']
E_cross_GHz = 0.5 * (EV_M[idx_bc, CANON['a']] + EV_M[idx_bc, CANON['b']]) / 1e3
print(f'  Crossing energy: {E_cross_GHz:.4f} GHz')

# Known crossing Bc values (from gate script refined output)
KNOWN_BC = [15155.9, 15158.0, 15160.4, 15160.4, 15162.4, 15167.2, 15296.9]
BC_CANONICAL = 15167.2  # G

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4 — Fig 1: Full Zeeman map 14000–16000 G, all 36 levels
# ---------------------------------------------------------------------------

# For the full map figure use the coarse scan BUT also need the range 14000+.
# The coarse scan starts at 0; select the B >= 14000 G portion for plotting.
mask_full = B_COARSE >= 14000.0
B_plot1 = B_COARSE[mask_full]
EV_plot1 = EV_C[mask_full]  # shape (n_B, 36)

fig1, ax1 = plt.subplots(figsize=(10, 6))

# Color by zero-field N character
color_map = {0: '#1f77b4', 1: '#d62728', 2: '#2ca02c'}  # blue, red, green
label_map = {0: 'N=0', 1: 'N=1', 2: 'N=2'}
plotted_labels = set()

for k in range(SIZE):
    n = DOMN0[k]
    col = color_map.get(n, 'gray')
    lbl = label_map.get(n, f'N={n}') if n not in plotted_labels else None
    ax1.plot(B_plot1, EV_plot1[:, k] / 1e3,  # MHz -> GHz
             color=col, lw=0.8, alpha=0.75, label=lbl)
    if lbl:
        plotted_labels.add(n)

# Shaded Karthein window
ax1.axvspan(WINDOW_LO, WINDOW_HI, alpha=0.12, color='gold',
            label='Karthein window\n15 050–15 300 G')

# Annotate canonical Bc — anchor at the actual crossing region: the rising
# N=0 (m_S=+1/2) line meets the descending N=1 (m_S=-1/2) line at
# (Bc, E_cross_GHz ~ +21 GHz), computed in Cell 3.
ax1.axvline(BC_CANONICAL, color='k', lw=1.2, ls='--', alpha=0.6)
ax1.annotate(f'$B_c$ = {BC_CANONICAL:.1f} G\n($E \\approx$ {E_cross_GHz:.1f} GHz)',
             xy=(BC_CANONICAL, E_cross_GHz),
             xytext=(BC_CANONICAL - 1100, E_cross_GHz + 28),
             fontsize=9, arrowprops=dict(arrowstyle='->', lw=0.9),
             ha='left')

ax1.set_xlabel('Magnetic field (G)', fontsize=12)
ax1.set_ylabel('Energy (GHz)', fontsize=12)
ax1.set_title(r'$^{29}$SiO$^+$ X$^2\Sigma^+$ Zeeman map — all 36 levels', fontsize=13)
ax1.set_xlim(14000, 16000)
ax1.legend(loc='upper left', fontsize=9, framealpha=0.8)
ax1.grid(True, alpha=0.25, lw=0.5)
fig1.tight_layout()

fig1_path = os.path.join(FIG_DIR, 'fig1_zeeman_map_full.png')
fig1.savefig(fig1_path, dpi=200, bbox_inches='tight')
print(f'Saved: {fig1_path}')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5 — Fig 2: Crossing zoom 15050–15300 G
#
# INDEX-SPACE NOTE: CANON['a']/CANON['b'] are column indices of the MEDIUM map
# (ordered by continuity from B~0). The zoom map starts at 15050 G, so
# ZeemanMap(order=True) assigns its columns by continuity from 15050 G — a
# DIFFERENT index space. Indexing EV_Z with medium-map indices picks wrong
# levels. Therefore select the canonical partners directly in the zoom map by
# maximum overlap with their high-field decoupled kets (gate-script method):
#   Partner A: |N=0, m_N=0, m_S=+1/2, m_I=-1/2>  -> rises  (+g_S muB/2)
#   Partner B: |N=1, m_N=0, m_S=-1/2, m_I=+1/2>  -> descends
# ---------------------------------------------------------------------------

# --- Decoupled-basis target kets ---
dqz = M.alt_q_numbers['decoupled']
selA = np.where(np.isclose(dqz['N'], 0) & np.isclose(dqz['M_N'], 0) &
                np.isclose(dqz['M_S'], +0.5) & np.isclose(dqz['M_I'], -0.5))[0]
selB = np.where(np.isclose(dqz['N'], 1) & np.isclose(dqz['M_N'], 0) &
                np.isclose(dqz['M_S'], -0.5) & np.isclose(dqz['M_I'], +0.5))[0]
assert len(selA) == 1 and len(selB) == 1, f'Target kets not unique: {selA}, {selB}'

# --- Select partners in ZOOM-map index space, 10 G below the crossing ---
idx_sel = int(np.argmin(np.abs(B_ZOOM - (BC_CANONICAL - 10.0))))
dec_sel = M.convert_evecs('decoupled', evecs=VEC_Z[idx_sel], verbose=False)  # (36, n_dec)
wA = dec_sel[:, selA[0]]**2
wB = dec_sel[:, selB[0]]**2
kA = int(np.argmax(wA))
kB = int(np.argmax(wB))
print(f'Zoom-map partner indices: kA={kA} (weight {wA[kA]:.3f}), kB={kB} (weight {wB[kB]:.3f})')
assert wA[kA] > 0.5 and wB[kB] > 0.5, 'Decoupled-ket overlap not dominant — selection failed'
assert kA != kB

# --- Verify the selected pair actually crosses in the window ---
gap_zoom = EV_Z[:, kB] - EV_Z[:, kA]
sc_zoom = np.where(np.diff(np.sign(gap_zoom)) != 0)[0]
assert len(sc_zoom) >= 1, 'Selected zoom pair does not cross in window!'
i_x = sc_zoom[0]
f_x = gap_zoom[i_x] / (gap_zoom[i_x] - gap_zoom[i_x+1])
Bc_zoom = B_ZOOM[i_x] + f_x * (B_ZOOM[i_x+1] - B_ZOOM[i_x])
print(f'Zoom-map crossing of selected pair: Bc = {Bc_zoom:.1f} G '
      f'(gate value {BC_CANONICAL:.1f} G)')
assert abs(Bc_zoom - BC_CANONICAL) < 5.0, 'Crossing field disagrees with gate value'

# --- Slopes of the two partners just below the crossing (MHz/G) ---
i1 = max(idx_sel - 20, 0)
slopeA = (EV_Z[idx_sel, kA] - EV_Z[i1, kA]) / (B_ZOOM[idx_sel] - B_ZOOM[i1])
slopeB = (EV_Z[idx_sel, kB] - EV_Z[i1, kB]) / (B_ZOOM[idx_sel] - B_ZOOM[i1])
print(f'Slopes near Bc: partner A = {slopeA:+.3f} MHz/G (expect ~+1.40), '
      f'partner B = {slopeB:+.3f} MHz/G (expect ~-1.40)')
assert slopeA > 1.0 and slopeB < -1.0, 'Slopes wrong sign — wrong levels selected'

# --- Crossing energy reference ---
idx_bc_zoom = int(np.argmin(np.abs(B_ZOOM - BC_CANONICAL)))
E_cross_zoom = 0.5 * (EV_Z[idx_bc_zoom, kA] + EV_Z[idx_bc_zoom, kB])
E_WIN = 300.0  # MHz

# Which zoom-map levels come within the energy window?
near_mask = np.any(np.abs(EV_Z - E_cross_zoom) < E_WIN, axis=0)  # shape (36,)

# N-character of ZOOM-map columns (dominant basis component at 15050 G;
# the case-(b) basis kets have definite N, so this remains a good label)
DOMN_Z = np.array([dom_N(M, VEC_Z[0, k]) for k in range(SIZE)])

fig2, ax2 = plt.subplots(figsize=(10, 6))

# Set ylim early so text placement works
YLIM2 = (-0.32, 0.32)
ax2.set_ylim(*YLIM2)

# Plot all near-crossing levels, color by N character (zoom-map index space)
for k in range(SIZE):
    if not near_mask[k] or k in (kA, kB):
        continue
    n = DOMN_Z[k]
    col = color_map.get(n, 'gray')
    ax2.plot(B_ZOOM, (EV_Z[:, k] - E_cross_zoom) / 1e3,
             color=col, lw=1.0, alpha=0.45)

# Canonical partner A: |N=0, J=1/2, F=0, M_F=0>, m_S=+1/2 at high field — RISES
ax2.plot(B_ZOOM, (EV_Z[:, kA] - E_cross_zoom) / 1e3,
         color='#1f77b4', lw=2.5,
         label=r'$|N{=}0,\ J{=}1/2,\ F{=}0,\ M_F{=}0\rangle$ ($m_S{=}{+}1/2$)',
         zorder=5)
# Canonical partner B: |N=1, J=3/2, F=2, M_F=0>, m_S=-1/2 — DESCENDS
ax2.plot(B_ZOOM, (EV_Z[:, kB] - E_cross_zoom) / 1e3,
         color='#d62728', lw=2.5, linestyle='--',
         label=r'$|N{=}1,\ J{=}3/2,\ F{=}2,\ M_F{=}0\rangle$ ($m_S{=}{-}1/2$)',
         zorder=5)

# Mark the crossing point itself
ax2.plot([Bc_zoom], [0.0], marker='o', ms=8, mfc='none', mec='k', mew=1.5, zorder=6)

# Mark known crossing Bc values
# Deduplicate for display (15160.4 appears twice)
bc_display = sorted(set(KNOWN_BC))
TEXT_Y = 0.28  # fixed y near top of plot
for bc in bc_display:
    ax2.axvline(bc, color='gray', lw=0.8, ls=':', alpha=0.7)
    label_txt = f'{bc:.1f}'
    if abs(bc - BC_CANONICAL) < 0.5:
        label_txt = f'{bc:.1f}*'
    ax2.text(bc, TEXT_Y, label_txt, rotation=90, fontsize=7.5,
             va='top', ha='right', color='#555555')

# Shaded Karthein window
ax2.axvspan(WINDOW_LO, WINDOW_HI, alpha=0.10, color='gold', label='Karthein window')

ax2.set_xlabel('Magnetic field (G)', fontsize=12)
ax2.set_ylabel(r'Energy $- E_c$ (GHz)', fontsize=12)
ax2.set_title(r'$^{29}$SiO$^+$ crossing zoom — Karthein window ($\pm$300 MHz)', fontsize=13)
ax2.set_xlim(ZOOM_LO, ZOOM_HI)
ax2.set_ylim(*YLIM2)
ax2.legend(fontsize=9, loc='upper right', framealpha=0.85)
ax2.grid(True, alpha=0.25, lw=0.5)
fig2.tight_layout()

fig2_path = os.path.join(FIG_DIR, 'fig2_crossing_zoom.png')
fig2.savefig(fig2_path, dpi=200, bbox_inches='tight')
print(f'Saved: {fig2_path}')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6 — Fig 3: Decoupled-basis composition vs B for canonical pair
#
# For each B in a set of sample points 0 to 16000 G, compute the dominant
# decoupled-basis weights |<m_N, m_S, m_I | psi(B)>|^2 for each canonical
# partner. Track the same states by continuity (use the same index tracking
# that the ZeemanMap order=True provides).
# ---------------------------------------------------------------------------

# Coarse composition scan: 80 log-spaced points from ~1 G to 16000 G
B_comp = np.concatenate([[1e-4], np.geomspace(1.0, 16000.0, 79)])
EV_comp, VEC_comp = zeeman_map(M, B_comp)

# Get decoupled basis
dq = M.alt_q_numbers['decoupled']  # dict with 'N','M_N','M_S','M_I','M_F' arrays

# Canonical partner indices (same as before, by zero-field tracking)
ka = CANON['a']
kb = CANON['b']

def get_decoupled_weights(m, vec):
    """Convert eigenvector to decoupled basis and return weights squared."""
    dv = m.convert_evecs('decoupled', evecs=np.array([vec]), verbose=False)[0]
    return dv**2

# Compute weights at each B for both partners
# Shape: (n_B, n_dec)
weights_a = np.array([get_decoupled_weights(M, VEC_comp[i, ka]) for i in range(len(B_comp))])
weights_b = np.array([get_decoupled_weights(M, VEC_comp[i, kb]) for i in range(len(B_comp))])

# Find the top-3 components at HIGH field (last B point) for each partner
top_a = np.argsort(-weights_a[-1])[:3]
top_b = np.argsort(-weights_b[-1])[:3]

def comp_label(dq, idx):
    return (f'$m_N$={dq["M_N"][idx]:+.0f}, '
            f'$m_S$={dq["M_S"][idx]:+.1f}, '
            f'$m_I$={dq["M_I"][idx]:+.1f}')

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

# Partner A
colors3 = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']
for j, idx in enumerate(top_a):
    ax3a.plot(B_comp, weights_a[:, idx],
              color=colors3[j % len(colors3)], lw=1.8,
              label=comp_label(dq, idx))
ax3a.set_xscale('log')
ax3a.set_xlabel('Magnetic field (G)', fontsize=11)
ax3a.set_ylabel(r'$|\langle m_N, m_S, m_I | \psi(B) \rangle|^2$', fontsize=11)
ax3a.set_title(r'Partner A: $|N{=}0,\ J{=}1/2,\ F{=}0,\ M_F{=}0\rangle$', fontsize=10)
ax3a.set_ylim(-0.02, 1.05)
ax3a.legend(fontsize=8.5, loc='center right')
ax3a.grid(True, alpha=0.25, lw=0.5)
ax3a.axvline(BC_CANONICAL, color='k', lw=1.0, ls='--', alpha=0.4, label=f'$B_c$={BC_CANONICAL:.0f} G')

# Partner B
for j, idx in enumerate(top_b):
    ax3b.plot(B_comp, weights_b[:, idx],
              color=colors3[j % len(colors3)], lw=1.8,
              label=comp_label(dq, idx))
ax3b.set_xscale('log')
ax3b.set_xlabel('Magnetic field (G)', fontsize=11)
ax3b.set_ylabel(r'$|\langle m_N, m_S, m_I | \psi(B) \rangle|^2$', fontsize=11)
ax3b.set_title(r'Partner B: $|N{=}1,\ J{=}3/2,\ F{=}2,\ M_F{=}0\rangle$', fontsize=10)
ax3b.set_ylim(-0.02, 1.05)
ax3b.legend(fontsize=8.5, loc='center right')
ax3b.grid(True, alpha=0.25, lw=0.5)
ax3b.axvline(BC_CANONICAL, color='k', lw=1.0, ls='--', alpha=0.4)

fig3.suptitle(r'$^{29}$SiO$^+$ canonical pair: decoupled-basis composition vs $B$',
              fontsize=12)
fig3.tight_layout()

fig3_path = os.path.join(FIG_DIR, 'fig3_composition_vs_B.png')
fig3.savefig(fig3_path, dpi=200, bbox_inches='tight')
print(f'Saved: {fig3_path}')
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7 — Summary table + Stark drive coupling
#
# Reproduces from gate script:
#   - 7-crossing table (Bc, parities, Ma, Mb, slope, mingap, DeltaM_F=0)
#   - Omega/2pi = 3.332 kHz at E = 6 V/cm
#   - xi = 2.66e-4
# ---------------------------------------------------------------------------

E_DRIVE = 6.0  # V/cm

def stark_operator(m):
    """V_E = H(1,0) - H(0,0) per hamiltonian_builders.py:99"""
    return m.H_function(1.0, 1e-9) - m.H_function(0.0, 1e-9)

VE = stark_operator(M)

# Off-crossing eigenvectors for canonical pair (Bc - 15 G)
va, vb = evec_offcrossing(M, CANON['Bc'], VEC_M[CANON['idx']], CANON['a'], CANON['b'])
va_p, vb_p = evec_offcrossing(M, CANON['Bc'], VEC_M[CANON['idx']],
                               CANON['a'], CANON['b'], offset=-15.0)

omega_m = abs(va @ (VE * E_DRIVE) @ vb)     # MHz
omega_p = abs(va_p @ (VE * E_DRIVE) @ vb_p)
omega = 0.5 * (omega_m + omega_p)
omega_khz = omega * 1e3
muE = M.parameters['muE']
bare = muE * E_DRIVE
xi = omega / bare

print('=' * 72)
print('CROSSING SUMMARY TABLE (all opposite-parity N0xN1, in window)')
print('=' * 72)
print(f'{"Bc [G]":>10} {"par_A":>6} {"par_B":>6} {"M_A":>5} {"M_B":>5}'
      f' {"slope [MHz/G]":>13} {"mingap [MHz]":>12} {"DM0":>5}')
print('-' * 72)
for c in IN_WINDOW:
    dm = 'yes' if c['Ma'] == c['Mb'] else 'no'
    canon_mark = ' <-- canonical' if abs(c['Bc'] - BC_CANONICAL) < 1.0 else ''
    print(f'{c["Bc"]:10.1f} {c["pa"]:+6d} {c["pb"]:+6d} {c["Ma"]:+5.0f} {c["Mb"]:+5.0f}'
          f' {c["slope"]:13.4f} {c["gmin"]:12.3f} {dm:>5}{canon_mark}')
print('=' * 72)
print(f'\nStark drive coupling at E = {E_DRIVE} V/cm (axial, ΔM_F=0):')
print(f'  Ω/2π = {omega_khz:.3f} kHz   [Karthein reported ~3.0 kHz]')
print(f'  ξ = Ω / (μ_E · E / h) = {xi:.2e}   [dimensionless spin-admixture suppression]')
print(f'  (reproduced from gate script: test_sio_crossing.py gate 5)')
print(f'\nConventions: B&C-audited; parity +(-1)^N verified; constants from Zhu 2022 / Knight 1985.')